<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.1A/MP8.1A_fins_resolved.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.1A · The Fins Resolved

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp81a.py', 'MP8.1A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.1A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp81a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp81a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP8.1A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex08 computed the temperature of a power module on an extruded heat sink with one simplification at its centre: the ten fins were **collapsed** into one effective coefficient on the underside of the base plate, $h_\mathrm{eff} = 322$ W/(m²K), by the textbook fin efficiency. That collapse is what every heat-sink datasheet and hand calculation uses, and it made the base plate a rectangle with an exact series solution.

It rests on two assumptions: that heat leaves the base evenly across its whole underside, and that every fin root sits at the temperature of a smooth fin side. In the real sink the heat leaves through ten roots only 2 mm wide, with bare metal between them, and the temperature under a root is lower than between two roots. How large is the error of the collapse, and where? An engineer who sizes a sink by the datasheet wants to know how far to trust it - and a design study that varies the fan speed and the module power wants the **resolved** answer many times.

This mini project computes the steady temperature of the **whole cross-section - base plate and fins - with the air's coefficient on every face in the air**, for any module power from 100 to 200 W and any air coefficient from 20 to 60 W/(m²K), with one network.

## A2 · The heat sink, and what is new against Ex08

The heat sink is Ex08's: an extruded aluminium sink with a 100 mm wide, 10 mm thick base plate and ten straight fins of 2 × 40 mm at a 10 mm pitch, cooled by a fan in an enclosure at 40 °C. The module sits on the middle 30 mm of the base. The cell below draws it and prints its data.

**What is new.** The domain is no longer the base plate alone but the **comb**: the base plate and the ten fins as one piece of metal. The air's coefficient $h$ acts on every face in the air - both sides of every fin, every fin tip, and the bare underside of the base between the fins. Nothing is collapsed.

**Two inputs.** The module's power $P$ (100 to 200 W) and the air's coefficient $h$ (20 to 60 W/(m²K): a slow and a fast fan; Ex08's 40 is in the middle; the range is **assumed**). The problem is linear in $P$, but not in $h$: $h$ changes how the heat divides between the fins and how far it travels along them.

In [ ]:
# The heat sink of Ex08, and its data
fig, ax = plt.subplots(figsize=(8, 3.6))
hs.draw_heatsink(ax, show_domain=False)                  # Ex08's drawing of the sink and the module
ax.set_title("the domain of MP8.1A: the whole comb, base plate and fins")
plt.show()

rows = [("base plate", f"{hs.W_BASE*1e3:.0f} mm wide, {hs.T_BASE*1e3:.0f} mm thick, {hs.DEPTH*1e3:.0f} mm deep"),
        ("aluminium 6061", f"k = {hs.K_AL:.0f} W/(m·K)"),
        ("fins", f"{hs.N_FINS} fins, {hs.T_FIN*1e3:.0f} × {hs.L_FIN*1e3:.0f} mm, pitch {hs.PITCH*1e3:.0f} mm"),
        ("air", f"{hs.T_AIR:.0f} °C, h = {mp.H_MIN:.0f}–{mp.H_MAX:.0f} W/(m²K) on every face in the air"),
        ("module", f"footprint {hs.MODULE[0]*1e3:.0f}–{hs.MODULE[1]*1e3:.0f} mm, power P = {mp.P_MIN:.0f}–{mp.P_MAX:.0f} W"),
        ("test cases", ", ".join(f"({P:.0f} W, {h:.0f} W/(m²K))" for P, h in mp.TESTS))]
for k, v in rows:
    print(f"{k:>15}  {v}")

**What you should see.** The sink in cross-section, the module on top, and the table of its data. Every value is Ex08's except the two ranges and the three test cases, which are inside the ranges but not at the values Ex08 used.

## A3 · The domain, the equation and its conditions

The domain is the comb in cross-section, per metre of depth: the base plate, $0 < x < W$ and $0 < z < t_b$, and below it ten fins, $-L_\mathrm{fin} < z < 0$, each 2 mm thick about its centre line at $x$ = 5, 15, ..., 95 mm. The unknown is the steady rise above the air, $\theta(x, z) = T - T_\mathrm{air}$, which depends also on $P$ and $h$.

$$\frac{\partial^2 \theta}{\partial x^2} + \frac{\partial^2 \theta}{\partial z^2} = 0 \quad \text{in the metal}$$

| where | condition | why it holds |
|---|---|---|
| module face, on the footprint ($z = t_b$) | $k\,\partial\theta/\partial z = \dfrac{P}{P_0}\,q''$ | the module's losses enter the base; $q'' = 50$ kW/m² at $P_0 = 150$ W |
| module face, off the footprint | $\partial\theta/\partial z = 0$ | still air above the base plate; its loss is small against the fins' |
| ends of the base ($x = 0$, $x = W$) | $\partial\theta/\partial x = 0$ | the two 10 mm ends lose little against 0.9 m of air faces per metre of depth |
| every face in the air: fin sides, fin tips, bare base | $-k\,\partial\theta/\partial n = h\,\theta$ | Newton's law of cooling; $n$ the outward normal |

The equation is **elliptic**: the temperature at every point depends on every condition at once. Heat crosses the 10 mm base in a few millimetres, turns into the roots, and travels down each fin, losing part of itself through the fin's sides on the way.

**The fin roots.** Where a fin meets the base there are two corners of 270° (re-entrant corners), twenty in all. At such a corner the temperature is continuous but its gradient is not bounded: it grows like $r^{-1/3}$ at a distance $r$ from the corner. Heat crowds round the corner on its way into the fin. Any solver - a grid or a network - pays for this: the grid converges more slowly there, and a smooth network cannot follow the corner exactly. Keep that in mind for part D.

In [ ]:
# The comb resolved on 0.5 mm cells: where the metal is, and the faces in the air
c = mp.Comb(0.5e-3)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[2, 1]))
ax[0].imshow(c.metal, origin="lower", cmap="Greys", extent=[0, 100, -40, 10], aspect="auto")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("z [mm]"); ax[0].set_title("the metal (dark) on 0.5 mm cells")
xa, za, nx, nz, w, _, _ = mp.sample_air_faces(3000, seed=1)
ax[1].scatter(xa * 1e3, za * 1e3, s=2, c=np.where(nz != 0, "tab:blue", "tab:orange"))
ax[1].set_xlim(30, 50); ax[1].set_ylim(-42, 11); ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("z [mm]")
ax[1].set_title("faces in the air: sides (orange), tips and bare base (blue)", fontsize=9)
plt.tight_layout(); plt.show()
xq, zq, wq, fq = mp.air_quadrature()
print(f"{c.n} metal cells of 0.5 mm; faces in the air: {wq.sum()*1e3:.0f} mm per mm of depth "
      f"({wq[fq == 1].sum()*1e3:.0f} mm on the fins, {wq[fq == 0].sum()*1e3:.0f} mm of bare base)")

**What you should see.** Left, the comb: the base plate on top and the ten fins below. Right, a close-up of two fins with points on the faces in the air. The air faces total 900 mm per mm of depth: 820 mm on the fins and 80 mm of bare base. That ratio is why the fins carry almost all the heat.

## A4 · Why there is no exact solution

Ex08's base plate has an exact solution because it is a **rectangle**: cosines across the base and known functions through its thickness fit every condition at once. The comb is not a rectangle. Its boundary turns twenty times at the roots, the bare base and the fin sides are different faces with different normals, and no separation of variables fits them all. No series or closed form exists. That is the normal case: almost every real part is a shape that no textbook formula fits.

The reference and the ground truth are therefore **numerical**: finite volumes on square cells (part C). The geometry has no strength parameter that tends continuously to Ex08: either the fins are resolved or they are not. The solver is checked instead in **Ex08's own limit**: with the fins removed and $h_\mathrm{eff}$ put on the base's underside, the same code must reproduce Ex08's exact series.

In [ ]:
# How far is Ex08's collapsed model from the resolved comb? At 150 W and h = 40 W/(m²K)
G = np.load("MP8.1A_truth.npz")
r = mp.reference(150.0, 40.0)                             # the resolved comb, finite volumes on 1 mm cells (part C)
col = mp.collapsed(r["x"], r["z"], 150.0, 40.0)           # Ex08's series in the base, the 1-D fin below each root
fig, ax = plt.subplots(figsize=(10, 3.2))
sc = ax.scatter(r["x"] * 1e3, r["z"] * 1e3, c=r["theta"] - col, s=6, cmap="RdBu_r", vmin=-0.6, vmax=0.6)
plt.colorbar(sc, label="resolved - collapsed [K]"); ax.set_xlabel("x [mm]"); ax.set_ylabel("z [mm]")
plt.tight_layout(); plt.show()
print(f"hottest point: resolved {float(G['conv_hottest_extrapolated']):.2f} K, collapsed (Ex08) {hs.hottest():.2f} K above the air")
print(f"largest difference anywhere: {np.abs(r['theta'] - col).max():.2f} K")

**What you should see.** The resolved comb is about 0.26 K hotter than the collapsed model over the whole base plate (52.71 K against Ex08's 52.45 K at the hottest point), the largest difference anywhere is 0.49 K, and the difference changes sign within a millimetre of the roots: colder right under each root, warmer between two roots. The collapse is good to about half a percent here - but the difference is ten times the accuracy asked below, so it cannot stand in for the resolved answer, and it is not uniform, so it cannot simply be corrected by a constant.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the hottest point, at the centre of the module's face | the test cases (115 W, 25), (165 W, 45), (195 W, 55 W/(m²K)), which you do not train on | within **0.15 K** of the ground truth |
| the field in the whole comb | the same cases, on the ground truth's 0.25 mm cells | within **0.2 K** everywhere |
| the heat leaving through the air faces | the same cases | within **1 %** of the module's power |
| the time to answer a new case online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over both ranges. Its online answer - the hottest point for a new $(P, h)$ - is what is timed against the reference model, which must solve the comb again for every new $h$. The training time is reported apart, as the offline cost.

**The agreed accuracy, and why it is 0.15 K.** The proposal asked for 0.1 K on the hottest point. The reference model (part C) is finite volumes on the coarsest grid that is useful: 1 mm cells, two across each fin. Its hottest point is 0.07 to 0.12 K off at the test cases, and its next coarser grid, 2 mm cells, is 3 K off because a fin is then one cell thick. 0.1 K would force the reference to 0.5 mm cells, four times the work, for an accuracy no fan's coefficient is known to. The agreed engineering accuracy is therefore **0.15 K** on the hottest point, **0.2 K** on the field and **1 %** on the heat out - what the 1 mm reference meets, and what the network must meet too.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of four inputs and returns one number:

$$(x,\ z,\ P,\ h)\ \longmapsto\ \theta(x, z; P, h)$$

the rise above the air at a point of the comb, for a module power and an air coefficient. Both are **inputs**, like the coordinates: one training covers every case in the ranges, and a new case is answered by evaluating the network, not by training again.

Two facts about the inputs are worth building on. The problem is **linear in $P$**: double the power and every rise doubles, so a network can return its answer at $P_0 = 150$ W and multiply it by $P/P_0$ exactly. And the coordinates have two very different scales: 100 mm across the base and 2 mm across a fin.

In [ ]:
# The scales of the problem
print(f"base: W = {hs.W_BASE*1e3:.0f} mm across, t_b = {hs.T_BASE*1e3:.0f} mm thick;  fins: {hs.T_FIN*1e3:.0f} mm thick, {hs.L_FIN*1e3:.0f} mm long")
print(f"temperature scale S = q_mean t_b / k = {mp.S_RES:.2f} K at 150 W  (the rise across the base's thickness)")
for h in (mp.H_MIN, 40.0, mp.H_MAX):
    m, eta, he = hs.fin_efficiency(h)
    print(f"h = {h:4.0f} W/(m²K): fin parameter m L = {m*hs.L_FIN:.2f}, fin efficiency {eta:.3f}, collapsed h_eff = {he:5.0f} W/(m²K)")

**What you should see.** The fin parameter $mL$ runs from 0.44 to 0.76 over the range of $h$: the fins are efficient (94 % to 84 %), so the temperature falls only a few kelvin along them, and the collapsed coefficient $h_\mathrm{eff}$ runs from 170 to 460 W/(m²K). The scale of the residuals below, $S = 0.9$ K, is the rise across the base plate's thickness at 150 W.

## B2 · The physics the network is trained with

The equation and its conditions become terms of the loss. Each residual is multiplied by a length and divided by $S\,P/P_0$, so that each is of order one whatever the case:

| loss term | where it is evaluated | its residual |
|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | points in the metal $(x, z, P, h)$ | $t_b^2\,(\theta_{xx} + \theta_{zz}) / (S P/P_0)$ |
| $\mathcal{L}_\mathrm{air}$ | points on the faces in the air, with their normal | $t_b\,(\partial\theta/\partial n + h\theta/k) / (S P/P_0)$ |
| $\mathcal{L}_\mathrm{module}$ | points on the module face | $t_b\,(\partial\theta/\partial z - q''(x)P/(P_0 k)) / (S P/P_0)$ |
| $\mathcal{L}_\mathrm{balance}$ (optional) | the whole air boundary at a few random cases | $100\,(Q_\mathrm{out}/P - 1)$: the heat out against the heat in, in per cent |

**What can be built in rather than trained.** Ex08 showed that a condition built into the network's form holds exactly. Here: the insulated ends hold for any weights if the network sees $x$ only through $\cos(n\pi x/W)$ and $\cos(2n\pi x/p)$ (with $p$ the fin pitch), whose slopes vanish at $x = 0$ and $x = W$; the module face holds if the network adds to a function that already carries the flux something whose $z$-slope vanishes at $z = t_b$. And there is a very good such function: **Ex08's collapsed model itself**, the exact series in the base and the textbook fin below each root, which is within about 0.6 K everywhere (part A). A network that learns only the correction to it has a small, smooth job - except at the roots.

**The energy balance.** Integrated over the comb, the equation says that the heat the module puts in leaves through the air faces: $Q_\mathrm{out} = D \oint h\,\theta\,ds = P$, with $D$ the 100 mm depth. A network that satisfied the equation and the conditions exactly would satisfy it too; a network that does not can drift by a constant everywhere, which the local residuals barely see. `mp.air_quadrature()` gives the points and lengths to compute $Q_\mathrm{out}$.

## B3 · The training points

`mp81a.py` provides samplers for each set of points:

* `mp.sample_interior(n)`: points in the metal, a quarter of them in the fins by default (`fin_share`), and half of each set within a few millimetres of the roots (`near_root`, exponential with mean 1.5 mm), where the field changes fastest;
* `mp.sample_air_faces(n)`: points on the faces in the air in proportion to their length, with their outward normal;
* `mp.sample_top(n)`: points on the module face, a tenth of them within 1.5 mm of the footprint's edges;
* `mp.sample_roots(n)`: points across the fin roots, for a design that treats the base and the fins apart and must join them.

Every point carries its own case $(P, h)$, drawn uniformly over the ranges. The three test cases are not placed on purpose: the network is scored where it was not fitted.

In [ ]:
# The training points
x, z, P, h = mp.sample_interior(4000, fin_share=0.25)
xa, za, nx, nz, w, Pa, ha = mp.sample_air_faces(1500)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].scatter(x * 1e3, z * 1e3, s=1, alpha=0.5, label="in the metal")
ax[0].scatter(xa * 1e3, za * 1e3, s=1, label="air faces")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("z [mm]"); ax[0].legend(fontsize=7, markerscale=5)
ax[1].hist(z * 1e3, bins=100); ax[1].set_xlabel("z [mm]"); ax[1].set_ylabel("points in the metal")
ax[2].scatter(P, h, s=1, alpha=0.3)
for Pt, htest in mp.TESTS:
    ax[2].plot(Pt, htest, "r*", ms=12)
ax[2].set_xlabel("P [W]"); ax[2].set_ylabel("h [W/(m²K)]")
plt.tight_layout(); plt.show()

**What you should see.** Left, the points cover the comb. Middle, the heights of the points in the metal: a peak on both sides of $z = 0$, the roots, over a floor in the base and in the fins. Right, the cases spread uniformly over the rectangle of $(P, h)$ with the three test cases as red stars.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: finite volumes on an engineering grid | the same scheme on grids fine enough to be converged |
| grid | square cells of 1 mm: 1 800 cells, two across each fin | 0.25 mm cells (28 800) for the fields, 0.125 mm (115 200) and Richardson extrapolation for the hottest point |
| its job | to be **timed** against the network: the online cost of a new case | to **score** the network and the reference |

Both use `mp81a.Comb`. The rest of this part explains the scheme, builds confidence in the ground truth, and measures the reference model's accuracy and time - the bar part D has to clear.

## C2 · The scheme, step by step

1. **Cells.** The cross-section is covered by square cells of side $\Delta$; a cell is metal if its centre is in the base or in a fin. With $\Delta$ dividing 1 mm, the comb's edges fall on cell faces exactly.
2. **Conduction between cells.** Between two neighbouring metal cells the heat flow per metre of depth is $k\,(\theta_j - \theta_i)$: the face is $\Delta$ long and the centres $\Delta$ apart.
3. **Faces in the air.** A metal face with air beyond it - a fin side, a fin tip, the bare base - passes $g\,\theta_i$ to the air, with $g = h\Delta/(1 + h\Delta/2k)$: half a cell of metal in series with the air's coefficient.
4. **The module.** The top cells under the footprint receive $q''\Delta\,P/P_0$; the rest of the top face and the base's two ends pass nothing.
5. **One sparse solve.** Every cell's balance - what flows in equals what flows out - is one row of a sparse linear system, solved directly. Only the air faces' diagonal depends on $h$, so the geometry and the conduction are built once, and a new case assembles one diagonal and solves.
6. **Reading the answer.** The hottest point, at the centre of the module's face, is the top cells' value plus half a cell of the flux's slope, interpolated to $x = W/2$.

In [ ]:
# The ground truth: its convergence study at 150 W and h = 40 W/(m²K), and Ex08's limit, from the truth file
G = np.load("MP8.1A_truth.npz")
print("cells          cells in the metal   hottest point   heat out")
for dx, n, hot, q in zip(G["conv_dx"], G["conv_cells"], G["conv_hottest"], G["conv_heat_out"]):
    print(f"{dx*1e3:6.3f} mm     {int(n):8d}        {hot:10.4f} K   {q:8.2f} W")
d = np.abs(np.diff(G["conv_hottest"]))
print("change of the hottest between grids:", np.round(d, 4), "K;  ratios", np.round(d[:-1] / d[1:], 2))
print(f"observed order {float(G['conv_order']):.2f};  Richardson extrapolation {float(G['conv_hottest_extrapolated']):.4f} K")
print(f"fins collapsed (Ex08's limit) against Ex08's exact series: {float(G['collapsed_error_K']):.4f} K at most")

**What you should see.** Four grids, each twice as fine. The change of the hottest point falls by 2.6 each time, not by 4: the observed order is 1.36, not the scheme's 2. That is the fin roots: at a 270° corner the field goes as $r^{2/3}$, and that singular part converges at a lower order on any grid. The Richardson extrapolation uses the observed order, and the 0.125 mm grid is within 0.006 K of it, about twenty-five times finer than the 0.15 K asked. The heat out equals the 150 W in on every grid: finite volumes conserve heat exactly, cell by cell. With the fins collapsed, the same code reproduces Ex08's exact series to 0.002 K: the code is right where the answer is known.

## C3 · The reference model: accuracy and time

The reference grid is the coarsest that is useful: 1 mm cells. The cell below scores it against the ground truth at the three test cases and times a new case. It runs on the CPU of the Colab runtime: a sparse direct solve of 1 800 unknowns gains nothing from a GPU.

In [ ]:
# The reference model against the ground truth, at the three test cases
worst = dict(hot=0.0, field=0.0, heat=0.0)
for P, h in mp.TESTS:
    key = f"{int(P)}W_{int(h)}h"
    r = mp.reference(P, h)
    truth = mp.truth_at(G["x"], G["z"], G["theta_" + key], float(G["dx"]), r["x"], r["z"])
    e_hot = abs(r["hottest"] - float(G["hottest_extrapolated_" + key]))
    e_field = np.abs(r["theta"] - truth).max()
    e_heat = abs(r["heat_out"] / P - 1) * 100
    worst = {k: max(worst[k], v) for k, v in dict(hot=e_hot, field=e_field, heat=e_heat).items()}
    print(f"P = {P:.0f} W, h = {h:.0f} W/(m²K): hottest {r['hottest']:.2f} K (truth {float(G['hottest_extrapolated_' + key]):.2f}), "
          f"off by {e_hot:.2f} K; field off by {e_field:.2f} K at most; heat out {r['heat_out']:.2f} W")
t_ref = middle(lambda: mp.reference(165.0, 45.0))        # the online cost of a new case (geometry built once)
t_build = middle(lambda: mp.Comb(1e-3).solve(165.0, 45.0))   # and with the geometry built again
print(f"\nreference model: hottest {worst['hot']:.2f} K, field {worst['field']:.2f} K, heat out {worst['heat']:.1e} % at worst")
print(f"online time of a new case: {t_ref*1e3:.2f} ms (with the geometry built again: {t_build*1e3:.2f} ms)")
REF = {"t_ref_s": t_ref, "worst_hot_K": worst["hot"], "worst_field_K": worst["field"]}   # kept for part E

**What you should see.** At the three test cases the reference model's hottest point is 0.07 to 0.12 K above the truth and its field at most 0.13 K off, inside the agreed 0.15 and 0.2 K; its heat out is exact, as for every finite-volume grid. A new case takes about 2 ms (about 5 ms if the geometry is built again). Its time is kept in `REF` for part E, which compares the network against it on this same runtime. The bar is low: a 1 800-cell sparse solve is fast, so the network's online time has to be well under a millisecond, or it has to answer many cases in one call.

In [ ]:
# The ground truth at 165 W, h = 45 W/(m²K): the field, and the hottest point against h at 150 W
key = "165W_45h"
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw=dict(width_ratios=[2, 1]))
sc = ax[0].scatter(G["x"][::4] * 1e3, G["z"][::4] * 1e3, c=G["theta_" + key][::4], s=1, cmap="inferno")
plt.colorbar(sc, ax=ax[0], label="θ at 165 W, h = 45 [K]"); ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("z [mm]")
hh = np.linspace(mp.H_MIN, mp.H_MAX, 9)
ax[1].plot(hh, [mp.reference(150.0, v)["hottest"] for v in hh], "o-", label="resolved (reference)")
ax[1].plot(hh, [float(mp.collapsed([hs.W_BASE / 2], [hs.T_BASE], 150.0, v)[0]) for v in hh], "--", label="collapsed (Ex08)")
ax[1].set_xlabel("h [W/(m²K)]"); ax[1].set_ylabel("hottest point at 150 W [K]"); ax[1].legend()
plt.tight_layout(); plt.show()

**What you should see.** Left, the field at 165 W: hottest under the module, a few kelvin cooler towards the ends of the base, and falling along every fin to its tip. Right, the hottest point against the air's coefficient: 95 K above the air with a slow fan, 39 K with a fast one - almost inversely proportional to $h$, which is why a network needs $h$ as an input, not only $P$. The collapsed model follows the resolved curve a fraction of a kelvin below it over the whole range.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else in this part is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(x, z, P, h)`:

| argument | meaning | shape |
|---|---|---|
| `x`, `z` | position in metres: $0 \le x \le 0.1$, $-0.04 \le z \le 0.01$ ($z < 0$ in the fins) | `(n, 1)` |
| `P` | the module's power in watts, 100 to 200 | `(n, 1)` |
| `h` | the air's coefficient in W/(m²K), 20 to 60 | `(n, 1)` |
| returns | the rise above the air $\theta$ in kelvin | `(n, 1)` |

Whatever you compute inside - features, built-in conditions, several sub-networks - stays inside `net`.

## D2 · The loss, given

`losses(net, seed)` draws fresh points with the samplers of part B and returns the three residual terms of part B's table: the equation in the metal, the faces in the air and the module face. The base's ends are not in it: build them in (part B, section 2), or add a term yourself. `balance_loss(net, seed)` is the energy balance at four random cases: optional, and in our experience it holds the level of the whole field.

**Every residual is divided by $S\,P/P_0$**, the rise across the base's thickness at the case's power (0.9 K at 150 W), and multiplied by $t_b$ or $t_b^2$, so that each is of order one.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

K, TB, S = hs.K_AL, hs.T_BASE, mp.S_RES                  # W/(m K), m, K: conductivity, base thickness, residual scale

def losses(net, seed, n_in=4000, n_air=1500, n_top=500):
    # in the metal: the steady heat equation, t_b² (θ_xx + θ_zz) = 0
    x, z, P, h = map(T, mp.sample_interior(n_in, seed, fin_share=0.25))
    x.requires_grad_(True); z.requires_grad_(True)       # autograd differentiates with respect to these
    th = net(x, z, P, h)
    r_pde = TB ** 2 * (d(d(th, x), x) + d(d(th, z), z)) / (S * P / mp.P0)
    # the faces in the air: ∂θ/∂n + (h/k) θ = 0, n the outward normal
    xa, za, nx, nz, w, Pa, ha = mp.sample_air_faces(n_air, seed + 1)
    xa, za, Pa, ha = T(xa).requires_grad_(True), T(za).requires_grad_(True), T(Pa), T(ha)
    ta = net(xa, za, Pa, ha)
    r_air = TB * (T(nx) * d(ta, xa) + T(nz) * d(ta, za) + ha / K * ta) / (S * Pa / mp.P0)
    # the module face, z = t_b: ∂θ/∂z = q''(x) (P/P0) / k on the footprint, 0 elsewhere
    xt, Pt, ht = mp.sample_top(n_top, seed + 2)
    zt = T(np.full(n_top, TB)).requires_grad_(True)
    tt = net(T(xt), zt, T(Pt), T(ht))
    q = T(np.where((xt > hs.MODULE[0]) & (xt < hs.MODULE[1]), hs.FLUX, 0.0) * Pt / mp.P0)
    r_top = TB * (d(tt, zt) - q / K) / (S * T(Pt) / mp.P0)
    return {"pde": (r_pde ** 2).mean(), "air": (r_air ** 2).mean(), "module": (r_top ** 2).mean()}

# the energy balance (optional; part B, section 2): the heat out through the air faces against P, in per cent
QX, QZ, QW, QF = map(T, mp.air_quadrature())             # midpoints, lengths (m) and fin flags of the air faces

def heat_out(net, P, h):
    """W through the air faces, for the 100 mm depth, at one case."""
    th = net(QX, QZ, torch.full_like(QX, P), torch.full_like(QX, h))
    return hs.DEPTH * (QW * h * th).sum()

def balance_loss(net, seed, n_cases=4):
    rng = np.random.default_rng(seed)
    r = [(heat_out(net, P, h) / P - 1) * 100 for P, h in zip(*mp.sample_cases(n_cases, rng))]
    return torch.stack(r).pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The four inputs, scaled to order one; or features built from them - cosines of $x$ whose slope vanishes at the ends, the distance below a root, the position across a fin;
* **what is built in.** The linearity in $P$; the insulated ends; the module face; Ex08's collapsed model as a starting answer that the network corrects (part B, section 2);
* **its size and shape.** One network for the whole comb, or one for the base and one for the fins joined at the roots. Remember why Part 2 uses tanh, not ReLU.

Two things the problem tells you, from parts A and B. The base plate is 100 mm wide and a fin 2 mm thick: one smooth function of $(x, z)$ has to change on both scales. And the twenty root corners carry a field that no smooth function follows exactly: wherever your network meets them, it keeps a residual there.

In [ ]:
# A quick score, given: the worst error of the hottest point and of the field (K) at the test cases
G = np.load("MP8.1A_truth.npz")
def quick_score(every=8):
    e_hot = e_field = 0.0
    for P, h in mp.TESTS:
        key = f"{int(P)}W_{int(h)}h"
        x, z = G["x"][::every], G["z"][::every]
        with torch.no_grad():
            f = net(T(x), T(z), T(np.full(x.size, P)), T(np.full(x.size, h))).cpu().numpy().ravel()
            hot = net(T([hs.W_BASE / 2]), T([hs.T_BASE]), T([P]), T([h])).item()
        e_hot = max(e_hot, abs(hot - float(G["hottest_extrapolated_" + key])))
        e_field = max(e_field, np.abs(f - G["theta_" + key][::every]).max())
    return e_hot, e_field

In [ ]:
# TODO 1 - your network. net(x, z, P, h) -> θ in K, every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, x, z, P, h):
        ...                                              # return θ, shape (n, 1)

torch.manual_seed(81)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` (and, if you choose, `balance_loss` and terms of your own). Decide the optimiser or optimisers, the number of steps, the learning rate and its schedule, the weight of each term, the number of points and whether they are drawn afresh each step. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

`quick_score()` below gives, in a second, the worst error of the hottest point and of the field at the three test cases, from the ground truth.

In [ ]:
# TODO 2 - your training
history = {"pde": [], "air": [], "module": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst error: hottest point %.3f K, field %.3f K" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the error of the field at one test case over the comb. Save each figure you want in the report as `MP8.1A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the hottest point, the field on the ground truth's cells and the heat out at the three test cases;
* the hottest point at 150 W for eleven values of $h$ from 20 to 60 W/(m²K), and at $h$ = 10 and 80, outside the range;
* the **online time** of a new case on this runtime - the hottest point from one call of `net` - and of 10 and 100 new cases in one call, against the reference model run 1, 10 and 100 times on the same runtime.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online time.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
G = np.load("MP8.1A_truth.npz")
out = {}
def hottest(P, h):
    """the network's hottest point (K), at the centre of the module face, for arrays of cases"""
    P, h = np.atleast_1d(P), np.atleast_1d(h)
    with torch.no_grad():
        return net(T(np.full(P.size, hs.W_BASE / 2)), T(np.full(P.size, hs.T_BASE)), T(P), T(h)).cpu().numpy().ravel()

for P, h in mp.TESTS:
    key = f"{int(P)}W_{int(h)}h"
    with torch.no_grad():
        out["field_" + key] = net(T(G["x"]), T(G["z"]), T(np.full(G["x"].size, P)), T(np.full(G["x"].size, h))).cpu().numpy().ravel()
        out["heat_out_" + key] = np.array(heat_out(net, P, h).item())
    out["hottest_" + key] = hottest(P, h)[0]
out["h_sweep"] = np.linspace(mp.H_MIN, mp.H_MAX, 11)
out["hottest_sweep"] = hottest(np.full(11, 150.0), out["h_sweep"])
out["h_outside"] = np.array([10.0, 80.0]); out["hottest_outside"] = hottest(np.full(2, 150.0), out["h_outside"])

# online time: the network against the reference model, on this runtime
timing = {}
rng = np.random.default_rng(0)
for k in (1, 10, 100):
    Ps, hs_ = rng.uniform(mp.P_MIN, mp.P_MAX, k), rng.uniform(mp.H_MIN, mp.H_MAX, k)
    timing[f"net_{k}"] = middle(lambda: hottest(Ps, hs_))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(p, v)["hottest"] for p, v in zip(Ps, hs_)], reps=3 if k == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP8.1A_net.pt"))   # the trained weights, beside the report
print(f"online, one new case: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.2f} ms "
      f"({timing['ref_1']/timing['net_1']:.1f} times)")
print(f"100 new cases: network {timing['net_100']*1e3:.2f} ms, reference {timing['ref_100']*1e3:.1f} ms "
      f"({timing['ref_100']/timing['net_100']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP8.1A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): hottest {REF['worst_hot_K']:.2f} K, field {REF['worst_field_K']:.2f} K, {REF['t_ref_s']*1e3:.2f} ms per new case")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the error of the hottest point at each test case (K), against `hottest_extrapolated_<case>`;
* the largest error of the field at each test case (K);
* the error of the heat out at each test case (%);
* the network's online time for one new case and for 100, the reference model's on the same runtime (all in `R`), and their ratios;
* the offline training time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP8.1A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the error of the field at the three test cases over the comb; the hottest point against $h$ at 150 W from your network, the reference model and the collapsed model (`R["h_sweep"]`, `R["hottest_sweep"]`); and the online times of the network and the reference for 1, 10 and 100 cases.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 10 are about your network and its result; 11 to 17 ask you to use it; 18 concludes. The next cell writes `MP8.1A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features you computed from them), layers, neurons and number of weights. Why this size and shape?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. Did you build Ex08's collapsed model into your network? Give your worst errors of the hottest point and of the field before training and after.": """
""",
    "4. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time in seconds.": """
""",
    "5. How did you choose the weights of the loss terms? Show the terms during training in one figure and say which was hardest to bring down.": """
""",
    "6. The hottest point at the three test cases: your error at each in kelvin. Are all within 0.15 K? How do they compare with the reference model's?": """
""",
    "7. The field at the three test cases: your largest error at each and where in the comb it sits. Why there?": """
""",
    "8. The heat leaving through the air faces at the three test cases, against the module's power: your error in per cent, and what holds it.": """
""",
    "9. The online time for one new case: your network against the reference model on the same runtime, in milliseconds, and the ratio. What does each spend its time on?": """
""",
    "10. Ask for 1, 10 and 100 new cases at once: give the three times of the network and of the reference model, and say why they grow so differently.": """
""",
    "11. By how much is Ex08's collapsed model off at the hottest point at the three test cases, against the ground truth? Is the collapse good enough for a datasheet?": """
""",
    "12. The hottest point against h at 150 W, from your network: fit it as a + b/h. What are a and b, and what does each mean for the sink?": """
""",
    "13. Outside the training range: your hottest point at h = 10 and 80 W/(m²K) against the reference model. How fast does the error grow outside the range?": """
""",
    "14. The observed order of the ground truth's convergence is 1.36, not 2. Explain why, and say what it means for a grid an engineer would choose.": """
""",
    "15. Train again with another seed. How much do the worst errors of the two runs differ? Is your result reproducible within the agreed accuracy?": """
""",
    "16. A design study wants the fan speed (h) that keeps the hottest point at 50 K above the air at 180 W. Find it with your network and check it with the reference model.": """
""",
    "17. If the fins were 50 mm long instead of 40 mm, could your network answer? What would you change to make it answer a range of fin lengths?": """
""",
    "18. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP8.1A — The Fins Resolved", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP8.1A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP8.1A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP8.1A_report.md into MP8.1A_report.pdf, with any figure
# saved as MP8.1A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP8.1A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP8.1A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP8.1A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP8.1A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass